# theta 扫描 · brps（PPO ⟷ ACH 单参数插值）

一键 notebook：配置 → 训练 → 可视化。与 `notebooks/ab_brps.ipynb`（mirror vs
league）互补——这里固定 mirror 自博弈，只扫**策略项**的插值权重 theta。

- **框架**：`src/mjai/algos/nn_updates.py::NNActorCriticUpdate` 是唯一的 NN 更新
  规则，策略损失为

      L_policy = (1 - theta) * L_ppo_clip + theta * L_ach

  `theta=0` 就是 PPO 截断代理，`theta=1` 就是论文忠实 ACH（Fu et al. ICLR 2022,
  Algorithm 2 / Eq. 29）。value 项与 entropy 项两端形式相同，**不参与插值**。
- **单因子**：优化器、优势处理、每批更新次数、梯度裁剪、网络结构全部共享，且默认
  取 ACH 侧（SGD 恒定 lr=1e-3、原始 GAE 优势、单次更新、无裁剪、trunk LayerNorm）。
  所以相邻 theta 之间**只差策略项**。若要评估 PPO 自己的最佳实践（Adam / 优势归一化
  / 多 epoch），那是另一个实验：改 `configs/exp/brps_ppo_mlp_mirror.yaml` 里注明的
  旋钮，且 theta>0 时会打印 `ACHFidelityWarning`。
- **指标**：nash_conv（exact）。BRPS 是**同时博弈**，`mjai.eval.nash` 只对非同时博弈算 exploitability，所以本图纵轴是 nash_conv（2p0s 下 exploitability = nash_conv/2，但仓库不做这个改名，图上标的就是实际算出来的量）。
- **输出目录**：`runs/nb_theta/brps/theta_<tag>/seed_N/`（含 `DONE` 标记；
  **重复执行训练 cell 会跳过已完成臂**，中断后续跑即可）。
- **注意**：最便宜的 cycling 博弈，PPO 端预期出现绕圈/平台，ACH 端预期收敛到 (1/16, 10/16, 5/16)——theta 扫描最容易看出差别的一个游戏（约 45–60 min）。

**直接 Runtime → Run All 即可**；想加深/加宽，改下一格参数后重跑（已完成的臂不会
被重训；要重训某臂需先删掉它的目录）。

In [ ]:
# === Parameters ===
GAME        = "brps"
THETAS      = [0.0, 0.25, 0.5, 0.75, 1.0]   # 0 = PPO, 1 = ACH
SEEDS       = [0, 1, 2]
TOTAL_ENV_STEPS = 60000   # per-arm budget (probe depth, not the paper's 1e7)
EVAL_EVERY      = 5000
FINAL_FRAC  = 0.1           # "final" = mean over the last 10% of x (D5 convention)
SHOW_TQDM   = True          # per-arm tqdm bar over env-steps

PROBE_GRAD_NORMS = True     # log the PPO and ACH terms' grad norms separately
"""Per-term gradient telemetry: train/grad_norm_{ppo,ach}[_scaled] and
train/grad_cos_ppo_ach. The two policy terms differ in gradient magnitude by
orders of magnitude, so theta is NOT the blend of influence -- this is what
tells you how much of a theta ranking is the operator and how much is the
effective step size. Costs two extra backward passes per update: measured
+8.5% on a full train round (Liar's Dice, CPU, batch 64). The update itself is
bit-identical either way."""

ON_STALE    = "error"       # "error" | "retrain" | "skip"
"""What to do with an arm that finished under a DIFFERENT config.

Arms are cached by a fingerprint of their resolved ExperimentConfig, not by
directory name, so raising TOTAL_ENV_STEPS (or changing the device, the eval
cadence, PROBE_GRAD_NORMS, any ACH knob) is detected instead of silently
"skipped".

    error    refuse that arm and print which knob changed. Nothing is
             deleted, and the other arms still run.
    retrain  DELETE the arm directory and train it again. Required rather
             than merely nice: a second TensorBoard event file in the same
             tb/ interleaves two runs into one curve.
    skip     reuse the mismatched result anyway.

DEVICE is part of the fingerprint, so flipping cpu <-> cuda marks every
finished arm stale. Use ON_STALE="skip" for one run if you just want the old
numbers back.
"""

DEVICE      = "cpu"         # "cpu" | "cuda" | None (= whatever the YAML says)
"""CPU is the default on purpose, and it is the FAST option here.

The rollout asks the policy for ONE decision at a time, so a 21->128->13
forward never fills a GPU: it is ~10 host<->device syncs of launch overhead
around a matmul that takes microseconds. Measured on Liar's Dice (RTX 3060 Ti):

    cpu    2809 env-steps/s      one policy call 241 us
    cuda    441 env-steps/s      one policy call 2110 us   (6.4x slower)

Set "cuda" only if you have raised the network width or batch size far enough
that the matmul dominates the launch overhead -- measure before assuming.
"""
from pathlib import Path
OUT_ROOT = Path("runs/nb_theta")

In [ ]:
# === Setup: import the probe machinery (no logic reimplemented here) ===
import sys
from pathlib import Path

REPO = Path.cwd()
if not (REPO / "tools" / "theta_probe.py").is_file():
    REPO = REPO.parent  # tolerate running from notebooks/
sys.path.insert(0, str(REPO / "tools"))

import arm_cache     # config-fingerprint cache (hit / stale / missing)
import policy_view   # final-policy view (rollout + mjai.eval.policy_table)
import theta_probe   # run_arm / arm_status / summarize / render_* helpers
from IPython.display import Image, display

def arm_kwargs():
    return dict(
        total_env_steps=TOTAL_ENV_STEPS,
        eval_every_env_steps=EVAL_EVERY,
        root=OUT_ROOT,
        device=DEVICE,
        probe_term_grad_norms=PROBE_GRAD_NORMS,
    )

def train_all():
    statuses, refused = [], []
    for theta in THETAS:
        for seed in SEEDS:
            label = f"theta={theta:<5g} seed={seed}"
            out = theta_probe.arm_dir(OUT_ROOT, GAME, theta, seed)
            st = theta_probe.arm_status(GAME, theta, seed, **arm_kwargs())
            action, why = arm_cache.resolve(st, ON_STALE, out)
            if action != "train":
                print(f"skip  {label}: {why}", flush=True)
                statuses.append((theta, seed, "cached" if action == "skip" else "REFUSED"))
                if action == "refuse":
                    refused.append(label)
                continue
            print(f"train {label}: {why}", flush=True)
            try:
                theta_probe.run_arm(
                    GAME, theta, seed, progress_bar=SHOW_TQDM, **arm_kwargs()
                )
                statuses.append((theta, seed, "done"))
            except Exception as e:  # keep going; report at the end
                statuses.append((theta, seed, f"FAILED: {type(e).__name__}: {e}"))
            print(f"      -> {statuses[-1][2]}", flush=True)
    if refused:
        print()
        print("=" * 72)
        print(f"{len(refused)} arm(s) REFUSED: finished under a different config.")
        print("Nothing was deleted. See the per-arm lines above for the changed knob,")
        print('then set ON_STALE="retrain" (rebuilds them) or "skip" (reuses them).')
        print("=" * 72)
    return statuses

print(f"{len(THETAS)} thetas x {len(SEEDS)} seeds = {len(THETAS) * len(SEEDS)} arms on {DEVICE}")

In [ ]:
# === Train (long cell: arms run sequentially; safe to re-run) ===
statuses = train_all()
for theta, seed, st in statuses:
    print(f"theta={theta:<5g} seed={seed}: {st}")

In [ ]:
# === Aggregate curves + per-theta results table ===
summary = theta_probe.summarize(OUT_ROOT, GAME, final_frac=FINAL_FRAC)
entry = summary.get(GAME, {})
print(f"metric: {entry.get('metric')}   final = mean over last {FINAL_FRAC:.0%} of x")
print()
for tag, arm in sorted(entry.get("thetas", {}).items(), key=lambda kv: kv[1]["theta"]):
    finals = {s: v for s, v in arm["final_per_seed"].items() if v is not None}
    vals = [round(v, 4) for v in finals.values()]
    mean = round(sum(finals.values()) / len(finals), 4) if finals else None
    print(f"theta={arm['theta']:<5g} final/seed={vals}  mean={mean}  done={len(arm['done'])}/{len(SEEDS)}")

In [ ]:
# === Figure 1: every theta's curve, overlaid (mean + min-max band) ===
fig1 = theta_probe.render_curves(summary, GAME, OUT_ROOT)
display(Image(filename=str(fig1))) if fig1 else print("no curves yet")

In [ ]:
# === Figure 2: final metric vs theta (error bar = min-max across seeds) ===
fig2 = theta_probe.render_theta_final(summary, GAME, OUT_ROOT)
display(Image(filename=str(fig2))) if fig2 else print("no finals yet")

In [ ]:
# === Diagnostic: gradient scale / per-term split / gate / clip, per theta ===
# The two policy terms differ in gradient magnitude by orders of magnitude (the
# ACH term carries an unbounded 1/pi_old), so the EFFECTIVE learning rate varies
# with theta. Read Figure 2 together with this panel before concluding that a
# theta is "better" — a monotone trend here means the ranking is partly a
# step-size effect, not purely a policy-operator effect.
#
# Six panels: total grad_norm; the PPO and ACH terms' own SCALED norms (what
# each contributes to the update, requires PROBE_GRAD_NORMS=True); the cosine
# between the two term gradients (< 0 = the terms disagree); ACH gate-off rate;
# PPO clip rate. The two term panels are the direct read on "who is driving
# this update" that the total norm cannot give you.
fig3 = theta_probe.render_telemetry(GAME, OUT_ROOT)
display(Image(filename=str(fig3))) if fig3 else print("no telemetry yet")

## 最终策略（每个 theta 学到了什么）

图 1/2 只说「离 Nash 多远」，不说「到底怎么打」。这一格把每个 theta 训练完的
**策略本身**物化出来。展示形式由游戏规模决定（实测枚举成本见
`src/mjai/eval/policy_table.py` 的模块 docstring）：

- **brps**（2 个信息集）：柱状图 + 与解析 NE (1/16, 10/16, 5/16) 的 TV 距离。
  这是最直观的一格——PPO 端（theta=0）绕圈时这里会明显偏离，ACH 端应该贴上去。
- **kuhn**（12）：完整表格。
- **liars_dice1**（24576）：动作边缘分布 + **按自博弈访问频率排序的 Top-K 信息集**，
  外加完整 CSV 落盘。

访问频率来自用该臂自己的策略自博弈 `POLICY_EPISODES` 局，按 observation 向量
join 回枚举出来的行。

In [ ]:
# === Final policy per theta ===
POLICY_PICK     = "best"   # "best" (SOTA snapshot) | "last" | "step_N"
POLICY_SEED     = 0        # which seed's arm to show
POLICY_EPISODES = 400      # self-play episodes used to rank info states (0 = skip)
POLICY_TOP_K    = 12       # rows in the printed table

import pandas as pd
from mjai.eval.policy_table import brps_nash_gap, to_records

policy_arms = [
    (f"theta={theta:g}", theta_probe.arm_dir(OUT_ROOT, GAME, theta, POLICY_SEED))
    for theta in THETAS
]
policy_arms = [(lab, run) for lab, run in policy_arms if (run / "checkpoints").is_dir()]

if not policy_arms:
    print("no trained arms yet")
else:
    fig_path, views, skipped = policy_view.render_arms(
        policy_arms, OUT_ROOT / "figs" / f"policy_{GAME}.png",
        checkpoint=POLICY_PICK, episodes=POLICY_EPISODES, player=0,
    )
    for lab, why in skipped.items():
        print(f"[{lab}] no policy table: {why}")
    if fig_path:
        display(Image(filename=str(fig_path)))
    if GAME == "brps":
        print("\nDistance to the analytic BRPS equilibrium (lower = closer):")
        for lab, view in views.items():
            probs, tv = brps_nash_gap(view)
            print(f"  {lab:<12s} P(R,P,S) = {probs.round(4)}   TV = {tv:.4f}")
    for lab, view in views.items():
        print(f"\n--- {lab} --- {view.checkpoint}")
        rows = view.top_rows(POLICY_TOP_K, player=0)
        df = pd.DataFrame(to_records(view, rows=rows)).set_index("info_state")
        display(df.style.format(precision=3, na_rep="-"))
        csv = OUT_ROOT / "figs" / f"policy_{GAME}_{policy_view.slug(lab)}.csv"
        print(f"    full table ({len(view.labels)} rows) -> "
              f"{policy_view.write_csv(view, csv)}")

## 解读指南

- **图 1（叠加曲线）**：同预算下谁低谁好。带宽是 3 个 seed 的 min–max，不是
  置信区间；单 seed 发散会把带撑开，这本身就是信息（cycling 博弈上 PPO 端更容易发生）。
- **图 2（theta–final）**：如果曲线单调下降，说明「越像 ACH 越好」；如果在中间出现
  极小值，说明混合策略项优于两个端点——这是本 notebook 唯一能给出的新结论，也是
  值得进一步查证的地方（先看图 3 排除步长效应）。
- **图 3（遥测，6 格）**：`train/grad_norm` 随 theta 的变化幅度决定了图 2 有多少是
  「策略算子」的功劳、多少是「有效步长」的功劳。中间两格
  `grad_norm_{ppo,ach}_scaled` 是**两项各自实际投进这次更新的梯度量**（总范数看不
  出这个），`grad_cos_ppo_ach` **小于 0 就说明两项在对着拉**——混合 theta 调不动的时候
  先看这一格。这三格需要 `PROBE_GRAD_NORMS=True`（本 notebook 默认开）。
  `gate_off_frac` 只在 theta>0 有值（ACH 门控关闭比例），`clip_frac` 只在 theta<1
  有值（PPO 截断比例）。
- **口径**：`final` 取 x 轴末 10% 的均值（docs/reproduce_report.md 的 D5 口径），
  不是最后一个点。要改口径就改参数格的 `FINAL_FRAC`。
- **与论文对照**：只有 `theta=1` 这一臂是论文忠实 ACH，可以和 docs/figs 里的数字化
  曲线比；其余 theta 都是本仓库自定义的插值实验，**不要**拿去对论文。
- **产物用法**：每臂的 `checkpoints/best/` 是该臂的 SOTA 快照，`uv run mjai-play`
  的 policy 列表会直接列出它。